# SMRD - longer-trained RoBERTa arm, stacked with the word-count arm (Colab, T4 GPU)

This notebook needs nothing from earlier runs: it works on a new Google account with an empty Drive.

Before running: **Runtime > Change runtime type > T4 GPU > Save**.

What it trains, on the same 5 folds as before:

* **lexical**: word counts + logistic regression (about a minute, no GPU used).
* **text_long**: the same RoBERTa source-tweet model as before, trained for up to 30 epochs instead of 6.
* **stack_text**: weighted average of the two.

The validation fold is kept exactly as before: it is never trained on. It alone picks the best epoch, the lexical
model's regularisation strength and the stack's weights. Each test fold is predicted once.

Only small result files are written to Drive (no saved models), so Drive space is not a concern. Finished folds are
skipped if you run a cell again. **Run the last cell before closing the tab or switching account.**

Earlier results for reference (pooled accuracy, from the previous account):

| | Twitter16 | Twitter15 | PHEME |
|---|---|---|---|
| RoBERTa text, 6 epochs | 0.688 | 0.736 | 0.867 |
| BiGCN | 0.804 | 0.797 | 0.858 |
| lexical | 0.848 | 0.840 | 0.871 |

In [ ]:
#@title 1. Dependencies and GPU check
import torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| GPU:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('NO GPU: choose Runtime > Change runtime type > T4 GPU, then run this cell again')
%pip install -q 'transformers==5.22.1' 'tokenizers==0.23.3' 'safetensors==0.8.0'     'huggingface_hub==2.4.0' 'pandas==3.1.0' 'scikit-learn==1.9.1' 'pytest==9.2.0'

In [ ]:
#@title 2. Mount Drive, get the text-long branch
import os, pathlib
from google.colab import drive
drive.mount('/content/drive')

BRANCH = 'text-long'
CKPT_ROOT = '/content/drive/MyDrive/smrd/checkpoints/cv'
os.makedirs(CKPT_ROOT, exist_ok=True)

PROJECT_ROOT = pathlib.Path('/content/smrd-reproduction')
if not PROJECT_ROOT.exists():
    !git clone -b {BRANCH} https://github.com/Miyonk/smrd-reproduction.git {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
os.chdir(PROJECT_ROOT)
!git log --oneline -1

In [ ]:
#@title 3. Unit tests (~2 min) - stop here if anything fails
!python -m pytest tests/test_encoder_pipeline.py tests/test_lexical_stack.py tests/test_cv.py -q

## twitter16

Run this one first and paste the output of 4c. Step b should take roughly 5-10 minutes on a T4.

In [ ]:
#@title 4a. twitter16: lexical arm, 5 folds (CPU)
!python -m src.run_cv train --dataset twitter16 --arm lexical --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 4b. twitter16: RoBERTa, up to 30 epochs, 5 folds (GPU)
!python -m src.run_cv train --dataset twitter16 --arm text_long --light --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 4c. twitter16: stack lexical + RoBERTa, then compare
!python -m src.run_cv stack --dataset twitter16 --members lexical text_long --tag text --ckpt-root {CKPT_ROOT}
!python -m src.run_cv compare --dataset twitter16 --ckpt-root {CKPT_ROOT}

## twitter15

Only after Twitter16 looks worthwhile. Step b takes about twice as long as for Twitter16.

In [ ]:
#@title 5a. twitter15: lexical arm, 5 folds (CPU)
!python -m src.run_cv train --dataset twitter15 --arm lexical --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 5b. twitter15: RoBERTa, up to 30 epochs, 5 folds (GPU)
!python -m src.run_cv train --dataset twitter15 --arm text_long --light --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 5c. twitter15: stack lexical + RoBERTa, then compare
!python -m src.run_cv stack --dataset twitter15 --members lexical text_long --tag text --ckpt-root {CKPT_ROOT}
!python -m src.run_cv compare --dataset twitter15 --ckpt-root {CKPT_ROOT}

## pheme

Optional and the most expensive (step b can take up to an hour of GPU). The 6-epoch model already matches the lexical arm here, so run this only if I ask for it.

In [ ]:
#@title 6a. pheme: lexical arm, 5 folds (CPU)
!python -m src.run_cv train --dataset pheme --arm lexical --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 6b. pheme: RoBERTa, up to 30 epochs, 5 folds (GPU)
!python -m src.run_cv train --dataset pheme --arm text_long --light --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 6c. pheme: stack lexical + RoBERTa, then compare
!python -m src.run_cv stack --dataset pheme --members lexical text_long --tag text --ckpt-root {CKPT_ROOT}
!python -m src.run_cv compare --dataset pheme --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 7. All comparisons again (saved to Drive)
import glob, shutil
!python -m src.run_cv compare --all-datasets --ckpt-root {CKPT_ROOT}
for p in glob.glob('results/cv/*.json'):
    shutil.copy(p, CKPT_ROOT)

In [ ]:
#@title 8. Flush Drive (run before closing the tab or switching account)
drive.flush_and_unmount()
print('Drive flushed and unmounted')